In [1]:
import pandas as pd

In [2]:
import numpy as np

In [3]:
#AnnoyIndex — yeh Spotify ka library hai. Approximate Nearest Neighbor (ANN) search karta hai fast.

In [4]:
import sys
print(sys.executable)

c:\Users\iaman\anaconda3\python.exe


In [5]:
%pip install annoy

  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> [22 lines of output]
      c:\Users\iaman\anaconda3\Lib\site-packages\setuptools\__init__.py:81: _DeprecatedInstaller: setuptools.installer and fetch_build_eggs are deprecated.
      !!
      
              ********************************************************************************
              Requirements should be satisfied by a PEP 517 installer.
              If you are using pip, you can try `pip install --use-pep517`.
              ********************************************************************************
      
      !!
        dist.fetch_build_eggs(dist.setup_requires)
      running bdist_wheel
      running build
      running build_py
      creating build
      creating build\lib.win-amd64-cpython-312
      creating build\lib.win-amd64-cpython-312\annoy
      copying annoy\__init__.py -> build\lib.win-amd64-cpython-312\annoy
      copying anno


  Using cached annoy-1.17.3.tar.gz (647 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Running setup.py clean for annoy
Failed to build annoy


In [6]:
import sys
print(sys.executable)

!pip --version

!python --version

c:\Users\iaman\anaconda3\python.exe


pip 24.0 from c:\Users\iaman\anaconda3\Lib\site-packages\pip (python 3.12)

Python 3.12.4


In [7]:
import sys
print(sys.version)

3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 15:03:56) [MSC v.1929 64 bit (AMD64)]


In [8]:
%pip install faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [9]:
import faiss

In [10]:
product=pd.read_csv('product_150k.csv')

In [11]:
product.head()

,p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,...,p151,p152,p153,p154,p155,p156,p157,p158,p159,product_id
0,0.057728,0.011255,-0.023404,0.092429,-0.020029,0.018735,0.003974,0.008556,-0.005657,0.016547,...,-0.021550,0.006544,0.006825,0.076608,-0.019186,-0.010564,0.000239,-0.043817,-0.097729,B01A5DU2TA
1,0.038126,0.004931,-0.022067,0.066682,0.009379,0.017764,0.019343,0.039371,0.001445,0.010448,...,0.004315,0.048992,0.021710,0.093479,-0.026546,-0.052332,0.004328,-0.019683,-0.112607,B07346RPKN
2,0.036354,0.016651,0.012105,0.096476,0.024649,-0.013194,0.018663,0.024827,-0.021811,-0.019258,...,0.017124,0.017427,0.026384,0.075618,0.015224,-0.049442,0.055570,-0.058420,-0.071863,B078TCVZ4V
3,0.046325,0.016477,-0.022943,0.018492,-0.025954,-0.005175,0.018342,0.041428,-0.006838,-0.008621,...,0.008416,0.010578,0.021611,0.051968,-0.002022,-0.055210,0.022411,-0.055057,-0.084781,B07BR3PPDR
4,0.042861,-0.012059,-0.051115,0.127641,0.026230,0.011335,0.004991,0.008443,0.037345,-0.019857,...,0.008416,0.010578,0.021611,0.051968,-0.002022,-0.055210,0.022411,-0.055057,-0.084781,B07H5C4P19


In [12]:
import os

size_mb = os.path.getsize("shopping_queries_dataset_products.parquet") / (1024*1024)
print(f"{size_mb:.2f} MB")

1057.49 MB


In [13]:
import pandas as pd

df = pd.read_parquet(
    "shopping_queries_dataset_products.parquet",
    columns=["product_id", "product_title"]
)

print(df.head())

   product_id                                      product_title
0  B079VKKJN7  11 Degrees de los Hombres Playera con Logo, Ne...
1  B079Y9VRKS          Camiseta Eleven Degrees Core TS White (M)
2  B07DP4LM9H  11 Degrees de los Hombres Core Pull Over Hoodi...
3  B07G37B9HP          11 Degrees Poli Panel Track Pant XL Black
4  B07LCTGDHY  11 Degrees Gorra Trucker Negro OSFA (Talla úni...


In [14]:
df_prod_embed=pd.merge(product,df.drop_duplicates(),on=['product_id'])

In [15]:
df_prod_embed.head()

,p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,...,p152,p153,p154,p155,p156,p157,p158,p159,product_id,product_title
0,0.057728,0.011255,-0.023404,0.092429,-0.020029,0.018735,0.003974,0.008556,-0.005657,0.016547,...,0.006544,0.006825,0.076608,-0.019186,-0.010564,0.000239,-0.043817,-0.097729,B01A5DU2TA,"Crocs Kids' Handle It Rain Boots , Candy Pink,..."
1,0.038126,0.004931,-0.022067,0.066682,0.009379,0.017764,0.019343,0.039371,0.001445,0.010448,...,0.048992,0.021710,0.093479,-0.026546,-0.052332,0.004328,-0.019683,-0.112607,B07346RPKN,"Hatley Kids' Little Classic Rain Boots, Pink &..."
2,0.036354,0.016651,0.012105,0.096476,0.024649,-0.013194,0.018663,0.024827,-0.021811,-0.019258,...,0.017427,0.026384,0.075618,0.015224,-0.049442,0.055570,-0.058420,-0.071863,B078TCVZ4V,Outee Rubber Kids Rain Boots
3,0.046325,0.016477,-0.022943,0.018492,-0.025954,-0.005175,0.018342,0.041428,-0.006838,-0.008621,...,0.010578,0.021611,0.051968,-0.002022,-0.055210,0.022411,-0.055057,-0.084781,B07BR3PPDR,Hope & Henry Girls' Red Milano Stitch Cardigan
4,0.042861,-0.012059,-0.051115,0.127641,0.026230,0.011335,0.004991,0.008443,0.037345,-0.019857,...,0.010578,0.021611,0.051968,-0.002022,-0.055210,0.022411,-0.055057,-0.084781,B07H5C4P19,Spring&Gege Youth Solid Full Zipper Hoodies So...


In [16]:
df_prod_embed['pid'] = range(0, df_prod_embed.shape[0])

In [17]:
df_prod_embed.reset_index(drop=True)

,p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,...,p153,p154,p155,p156,p157,p158,p159,product_id,product_title,pid
0,0.057728,0.011255,-0.023404,0.092429,-0.020029,0.018735,0.003974,0.008556,-0.005657,0.016547,...,0.006825,0.076608,-0.019186,-0.010564,0.000239,-0.043817,-0.097729,B01A5DU2TA,"Crocs Kids' Handle It Rain Boots , Candy Pink,...",0
1,0.038126,0.004931,-0.022067,0.066682,0.009379,0.017764,0.019343,0.039371,0.001445,0.010448,...,0.021710,0.093479,-0.026546,-0.052332,0.004328,-0.019683,-0.112607,B07346RPKN,"Hatley Kids' Little Classic Rain Boots, Pink &...",1
2,0.036354,0.016651,0.012105,0.096476,0.024649,-0.013194,0.018663,0.024827,-0.021811,-0.019258,...,0.026384,0.075618,0.015224,-0.049442,0.055570,-0.058420,-0.071863,B078TCVZ4V,Outee Rubber Kids Rain Boots,2
3,0.046325,0.016477,-0.022943,0.018492,-0.025954,-0.005175,0.018342,0.041428,-0.006838,-0.008621,...,0.021611,0.051968,-0.002022,-0.055210,0.022411,-0.055057,-0.084781,B07BR3PPDR,Hope & Henry Girls' Red Milano Stitch Cardigan,3
4,0.042861,-0.012059,-0.051115,0.127641,0.026230,0.011335,0.004991,0.008443,0.037345,-0.019857,...,0.021611,0.051968,-0.002022,-0.055210,0.022411,-0.055057,-0.084781,B07H5C4P19,Spring&Gege Youth Solid Full Zipper Hoodies So...,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147668,0.052873,-0.033392,-0.078980,0.077770,-0.016582,-0.000300,0.038925,0.009066,0.018365,-0.021745,...,0.002387,0.040072,-0.002007,-0.018031,0.011338,-0.050491,-0.088851,B08DQ2HNNB,Jurassic Park T-Rex Skull 1-Ply Reusable Face ...,147668
147669,0.008746,-0.083402,0.014344,-0.000829,0.019679,-0.001146,-0.014921,0.018712,0.029838,-0.006820,...,0.002387,0.040072,-0.002007,-0.018031,0.011338,-0.050491,-0.088851,B07DQCL946,The Jurassic Games,147669
147670,0.026175,-0.062091,-0.011162,0.036014,-0.043672,0.024141,0.013321,-0.015863,-0.006382,0.024280,...,0.026201,0.039388,0.063526,0.018322,0.059560,-0.036569,-0.126742,B07FPT34TS,Jurassic+World%3a+Fallen+Kingdom+Velociraptor+...,147670
147671,0.050990,-0.006569,-0.058267,0.085894,0.017769,0.008434,0.016887,0.010712,-0.017868,-0.012275,...,0.017682,0.054370,-0.040309,-0.028475,0.002260,-0.008909,-0.113362,B07BX7PCZX,Jurassic Park 8-Bit Logo Unisex Youth Pull-Ove...,147671


In [18]:
#Hum pid aur qid isliye bana rahe hain taaki embeddings ko Annoy/FAISS index me store karke nearest-neighbor search kar sakein aur baad me un integer IDs ko original products aur queries se map kar sakein.
#get_nns_by_vector(query_embedding, k) query vector ko sab product vectors se compare karta hai aur jo embeddings sabse similar hoti hain unke pid return karta hai. Baad me hum un pid ko DataFrame se map karke actual product names nikal lete hain.

In [19]:
df_query_embedding = pd.read_csv('query_150k.csv')

In [20]:
df_query_embedding['qid'] = range(0, df_query_embedding.shape[0])

In [21]:
df_prod_embed

,p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,...,p153,p154,p155,p156,p157,p158,p159,product_id,product_title,pid
0,0.057728,0.011255,-0.023404,0.092429,-0.020029,0.018735,0.003974,0.008556,-0.005657,0.016547,...,0.006825,0.076608,-0.019186,-0.010564,0.000239,-0.043817,-0.097729,B01A5DU2TA,"Crocs Kids' Handle It Rain Boots , Candy Pink,...",0
1,0.038126,0.004931,-0.022067,0.066682,0.009379,0.017764,0.019343,0.039371,0.001445,0.010448,...,0.021710,0.093479,-0.026546,-0.052332,0.004328,-0.019683,-0.112607,B07346RPKN,"Hatley Kids' Little Classic Rain Boots, Pink &...",1
2,0.036354,0.016651,0.012105,0.096476,0.024649,-0.013194,0.018663,0.024827,-0.021811,-0.019258,...,0.026384,0.075618,0.015224,-0.049442,0.055570,-0.058420,-0.071863,B078TCVZ4V,Outee Rubber Kids Rain Boots,2
3,0.046325,0.016477,-0.022943,0.018492,-0.025954,-0.005175,0.018342,0.041428,-0.006838,-0.008621,...,0.021611,0.051968,-0.002022,-0.055210,0.022411,-0.055057,-0.084781,B07BR3PPDR,Hope & Henry Girls' Red Milano Stitch Cardigan,3
4,0.042861,-0.012059,-0.051115,0.127641,0.026230,0.011335,0.004991,0.008443,0.037345,-0.019857,...,0.021611,0.051968,-0.002022,-0.055210,0.022411,-0.055057,-0.084781,B07H5C4P19,Spring&Gege Youth Solid Full Zipper Hoodies So...,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147668,0.052873,-0.033392,-0.078980,0.077770,-0.016582,-0.000300,0.038925,0.009066,0.018365,-0.021745,...,0.002387,0.040072,-0.002007,-0.018031,0.011338,-0.050491,-0.088851,B08DQ2HNNB,Jurassic Park T-Rex Skull 1-Ply Reusable Face ...,147668
147669,0.008746,-0.083402,0.014344,-0.000829,0.019679,-0.001146,-0.014921,0.018712,0.029838,-0.006820,...,0.002387,0.040072,-0.002007,-0.018031,0.011338,-0.050491,-0.088851,B07DQCL946,The Jurassic Games,147669
147670,0.026175,-0.062091,-0.011162,0.036014,-0.043672,0.024141,0.013321,-0.015863,-0.006382,0.024280,...,0.026201,0.039388,0.063526,0.018322,0.059560,-0.036569,-0.126742,B07FPT34TS,Jurassic+World%3a+Fallen+Kingdom+Velociraptor+...,147670
147671,0.050990,-0.006569,-0.058267,0.085894,0.017769,0.008434,0.016887,0.010712,-0.017868,-0.012275,...,0.017682,0.054370,-0.040309,-0.028475,0.002260,-0.008909,-0.113362,B07BX7PCZX,Jurassic Park 8-Bit Logo Unisex Youth Pull-Ove...,147671


In [22]:
df_prod_embed['pid'] = range(0, df_prod_embed.shape[0])

In [23]:
query_tower_input_dim = 32

In [24]:
product_tower_input_dim = 32 * 5

In [25]:
%pip install faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [26]:
import faiss
import numpy as np

In [27]:
df_query_embedding.head()

,q0,q1,q2,q3,q4,q5,q6,q7,q8,q9,...,q24,q25,q26,q27,q28,q29,q30,q31,query,qid
0,0.016756,-0.072765,0.010259,0.048307,0.074665,0.084850,0.022835,-0.066671,0.039883,-0.027148,...,0.011363,0.015675,0.018649,0.025668,0.000701,-0.006060,-0.078536,-0.026082,child proof cabinet locks,0
1,-0.007341,-0.010955,0.059960,0.071282,0.022258,0.032136,0.042009,-0.041242,-0.009459,-0.017514,...,-0.014555,-0.002024,0.062305,0.020665,-0.005689,0.032699,-0.045051,-0.096923,ankle stockings for women sheer,1
2,0.027613,0.019582,-0.032154,0.018755,0.037383,-0.025616,0.024898,-0.025434,-0.047236,0.022970,...,-0.007898,-0.011111,0.059078,0.020216,0.017000,-0.034270,-0.007410,-0.026015,gluten free snacks,2
3,0.021799,-0.018171,0.005133,0.049379,0.021239,-0.004974,0.022700,-0.024240,-0.014328,0.018335,...,0.003084,-0.041141,0.033879,-0.002729,-0.015864,0.026388,-0.021622,-0.083685,hair geow,3
4,0.036940,-0.077912,-0.036468,0.036546,0.121124,0.009458,0.050458,-0.060216,-0.033329,-0.018182,...,0.049342,0.031401,0.065255,0.086359,-0.056535,-0.013196,-0.023676,-0.070366,"1 by one, amplified, outdoor hdtv antenna",4


In [28]:
embedding_cols = [f"q{i}" for i in range(32)]

In [29]:
embedding_cols 

['q0',
 'q1',
 'q2',
 'q3',
 'q4',
 'q5',
 'q6',
 'q7',
 'q8',
 'q9',
 'q10',
 'q11',
 'q12',
 'q13',
 'q14',
 'q15',
 'q16',
 'q17',
 'q18',
 'q19',
 'q20',
 'q21',
 'q22',
 'q23',
 'q24',
 'q25',
 'q26',
 'q27',
 'q28',
 'q29',
 'q30',
 'q31']

In [30]:
query_vectors = df_query_embedding[
    embedding_cols
].values

In [31]:
query_vectors.shape

(70756, 32)

In [32]:
query_vectors = df_query_embedding[
    embedding_cols
].values.astype('float32')

In [33]:
query_vectors 

array([[ 0.0167561 , -0.0727649 ,  0.01025946, ..., -0.00605989,
        -0.07853622, -0.02608193],
       [-0.00734055, -0.01095452,  0.05995964, ...,  0.03269918,
        -0.04505076, -0.09692273],
       [ 0.02761326,  0.01958188, -0.03215422, ..., -0.03426971,
        -0.00740957, -0.02601549],
       ...,
       [ 0.01690196, -0.03444054, -0.00344502, ...,  0.00973551,
        -0.01526183, -0.11378161],
       [ 0.0139008 ,  0.00348879,  0.03569293, ...,  0.0454612 ,
        -0.03942469, -0.11470607],
       [-0.0013913 , -0.01291055,  0.02124339, ..., -0.00735211,
        -0.01704622, -0.0812083 ]], shape=(70756, 32), dtype=float32)

In [34]:
index = faiss.IndexFlatL2(32)

In [35]:
index

<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x0000021A72246A30> >

In [36]:
index.add(query_vectors)

In [37]:
index

<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x0000021A72246A30> >

In [38]:
query = query_vectors[0].reshape(1, -1)

In [39]:
query

array([[ 0.0167561 , -0.0727649 ,  0.01025946,  0.04830723,  0.0746651 ,
         0.08485038,  0.02283522, -0.06667107,  0.03988254, -0.02714804,
         0.06233998, -0.04770579, -0.21258827,  0.01908997, -0.01554929,
        -0.09674209,  0.0146773 ,  0.01490625, -0.04414658,  0.00855378,
        -0.05610645,  0.01758992, -0.05080061, -0.00214545,  0.0113626 ,
         0.01567484,  0.0186487 ,  0.02566776,  0.00070053, -0.00605989,
        -0.07853622, -0.02608193]], dtype=float32)

In [40]:
distances, indices = index.search(
    query,
    5
)

In [41]:
indices

array([[    0, 54574,  1776, 44038, 33876]])

In [42]:
distances

array([[0.        , 0.00625084, 0.00906168, 0.00979975, 0.0100304 ]],
      dtype=float32)

In [43]:
for idx in indices[0]:
    print(df_query_embedding.iloc[idx]['query'])

child proof cabinet locks
child locks for cabinets
cabinet locks
cabinet handles
magnetic child safety cabinet locks


In [44]:
df_dataset = pd.merge(
    pd.read_csv('dataset_150k.csv'), 
    df_prod_embed[['product_id','product_title']].drop_duplicates(), 
    on=['product_id']
)

In [45]:
df_dataset['binary_label'] = df_dataset['esci_label'].apply(
    lambda x: 1 if x == 'E' else 0
)

In [46]:
query_tower_cols = [f"q{i}" for i in range(32)]

In [47]:
product_tower_cols = [f"p{i}" for i in range(160)]

In [48]:
train_data = df_dataset[df_dataset['split'] != 'test']
val_data   = df_dataset[df_dataset['split'] == 'test']

train_inputs = [
    np.array(train_data[query_tower_cols]),    # shape: (N, 32)
    np.array(train_data[product_tower_cols])   # shape: (N, 160)
]

val_inputs = [
    np.array(val_data[query_tower_cols]),
    np.array(val_data[product_tower_cols])
]

In [49]:
print(df_dataset.shape)
print(df_dataset.columns[:20])
print(df_dataset.columns[-20:])

(157097, 198)
Index(['query', 'product_id', 'esci_label', 'split', 'q0', 'q1', 'q2', 'q3',
       'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10', 'q11', 'q12', 'q13', 'q14',
       'q15'],
      dtype='object')
Index(['p142', 'p143', 'p144', 'p145', 'p146', 'p147', 'p148', 'p149', 'p150',
       'p151', 'p152', 'p153', 'p154', 'p155', 'p156', 'p157', 'p158', 'p159',
       'product_title', 'binary_label'],
      dtype='object')


In [50]:
embedding_dim = 16

In [51]:
%pip install wrapt

Note: you may need to restart the kernel to use updated packages.


In [52]:
%pip install h5py wrapt

In [53]:
from tensorflow.keras.layers import Input, Dense, Lambda, Dot
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K

In [54]:
input_query = Input(shape=(32,))

Input(shape=(32,)) — query ka input layer, 32 floats lega

In [55]:
final_query_embedding = Dense(
    16,
    activation='linear'
)(input_query)

# Iske andar actually ye ho raha hai:
# Output = Input × W + b
# jahan:
# Input = 32 values
# W = learnable weight matrix
#     (32 × 16)

# b = bias vector

Iske andar actually ye ho raha hai:

Output = Input × W + b

jahan:

Input = 32 values

W = learnable weight matrix
    (32 × 16)

b = bias vector

In [56]:
#L2 Normalization

normalized_query = Lambda(
    lambda x: K.l2_normalize(x, axis=-1),
    name='normalize_query'
)(final_query_embedding)

Cosine similarity nikalne ke liye.

Cosine similarity tab best kaam karti hai jab vectors normalized ho.
FAISS / Retrieval systems me millions of vectors hote hain.

Normalized vectors ke saath:

Search Fast
Similarity Stable
Training Better

K.l2_normalize(x)

ka matlab:

Vector ki length ko 1 bana do taaki model magnitude ki jagah semantic direction compare kare aur cosine similarity sahi tarike se kaam kare
Dot Product Range

Normalized vectors ke liye:

-1 to +1

range hoti hai.

Meaning:

1   → same direction

0   → unrelated

-1  → opposite direction

L2 normalization converts embeddings into unit vectors. Once both query and product embeddings have unit length, their dot product becomes equivalent to cosine similarity. Cosine similarity measures semantic alignment independent of vector magnitude, making it well-suited for retrieval and recommendation systems.

In [57]:
final_product_embedding = Dense(
    16,
    activation='linear',
    name='embedding_layer_product'
)(input_product)

normalized_product = Lambda(
    lambda x: K.l2_normalize(x, axis=-1),
    name='normalize_product'
)(final_product_embedding)

NameError: name 'input_product' is not defined

In [58]:
input_product = Input(shape=(160,), name='input_product')

In [59]:
final_product_embedding = Dense(
    16,
    activation='linear',
    name='embedding_layer_product'
)(input_product)

In [60]:
normalized_product = Lambda(
    lambda x: K.l2_normalize(x, axis=-1),
    name='normalize_product'
)(final_product_embedding)

In [61]:
cosine_similarity = Dot(
    axes=1,
    normalize=True
)([normalized_product, normalized_query])

In [62]:
cosine_similarity

<KerasTensor shape=(None, 1), dtype=float32, sparse=False, ragged=False, name=keras_tensor_9>

In [63]:
#“Ek neural network bana jo 2 inputs lega aur 1 output dega”
model = Model(
    inputs=[input_query, input_product],
    outputs=cosine_similarity
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_product       │ (None, 160)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer         │ (None, 32)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_layer_pr… │ (None, 16)        │      2,576 │ input_product[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 16)        │        528 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalize_product   │ (None, 16)        │          0 │ embedding_layer_… │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalize_query     │ (None, 16)        │          0 │ dense[0][0]       │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot (Dot)           │ (None, 1)         │          0 │ normalize_produc… │
│                     │                   │            │ normalize_query[… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,104 (12.12 KB)

 Trainable params: 3,104 (12.12 KB)

 Non-trainable params: 0 (0.00 B)

In [64]:
import numpy as np

q = np.random.rand(1, 32)
p = np.random.rand(1, 160)

model.predict([q, p])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step


array([[-0.10414285]], dtype=float32)

Query (32-d) + Product (160-d)
        ↓
16-d embedding each
        ↓
cosine similarity score

input_product → (None, 160)
input_layer   → (None, 32)
Meaning:
None = batch size (kitne bhi examples ek saath)
160 = product features
32 = query features

2. Dense layers (REAL learning happens here)
Product tower:
embedding_layer_product → (None, 16)
Params: 2,576

👉 meaning:

160 features → 16 learned features
Query tower:
dense → (None, 16)
Params: 528

👉 meaning:

32 features → 16 learned features

Why product has more parameters?

Because:

160 × 16 weights = 2560 + bias ≈ 2576
32 × 16 weights = 512 + bias ≈ 528

👉 product tower is bigger → more information → more weights

3. Normalization layers
normalize_product → (None, 16)
normalize_query   → (None, 16)
Meaning:
vector length = 1
only direction matters

👉 magnitude removed

4. Dot layer (FINAL OUTPUT)
dot → (None, 1)
Meaning:
Query vector (16)
      ·
Product vector (16)
      ↓
Single similarity score

Example:

0.92 → very relevant
0.10 → not relevant
-0.3 → opposite

In [65]:
from tensorflow.keras.optimizers import Adam

In [66]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

1. optimizer = Adam

👉 model ka learning engine

galti → weight update

2. loss = binary_crossentropy 👉 model ko punish karta hai wrong prediction par

3. metrics = accuracy

Sirf monitoring ke liye:

kitne sahi predictions hue
Step 2: ModelCheckpoint kya karta hai?

In [67]:
from tensorflow.keras.callbacks import ModelCheckpoint

In [73]:
model_checkpoint=ModelCheckpoint(
    'best_model_weights.h5',
    monitor='val_loss',
    save_best_only=True,
    mode='min'
)
#👉 “training ke dauran best model save karo”

In [74]:
train_labels = train_data['binary_label'].values
val_labels = val_data['binary_label'].values

In [75]:
#⚙️ Step 3: training start
history = model.fit(
    train_inputs,
    train_labels,
    epochs=100,
    batch_size=64,
    validation_data=(val_inputs, val_labels),
    callbacks=[model_checkpoint],
    class_weight=dict(enumerate([1, 2]))
)

Epoch 1/100
2120/2138 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5276 - loss: 1.0120

2138/2138 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.5282 - loss: 1.0108 - val_accuracy: 0.7949 - val_loss: 0.5173
Epoch 2/100
2122/2138 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7081 - loss: 0.7317

2138/2138 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.7079 - loss: 0.7319 - val_accuracy: 0.8443 - val_loss: 0.4746
Epoch 3/100
2138/2138 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.6774 - loss: 0.7500 - val_accuracy: 0.7970 - val_loss: 0.5094
Epoch 4/100
2135/2138 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7062 - loss: 0.7262

2138/2138 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.7062 - loss: 0.7262 - val_accuracy: 0.8415 - val_loss: 0.4431
Epoch 5/100
2138/2138 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.7456 - loss: 0.6722 - val_accuracy: 0.8007 - val_loss: 0.4988
Epoch 6/100
2138/2138 ━━━━━━━━━━━━━━━━━━━━ 22s 6ms/step - accuracy: 0.7605 - loss: 0.6587 - val_accuracy: 0.8031 - val_loss: 0.4939
Epoch 7/100
2138/2138 ━━━━━━━━━━━━━━━━━━━━ 16s 7ms/step - accuracy: 0.7328 - loss: 0.6942 - val_accuracy: 0.8352 - val_loss: 0.4760
Epoch 8/100
2138/2138 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.6557 - loss: 0.7694 - val_accuracy: 0.8057 - val_loss: 0.4831
Epoch 9/100
2138/2138 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.7619 - loss: 0.6554 - val_accuracy: 0.8221 - val_loss: 0.4592
Epoch 10/100
2138/2138 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.7633 - loss: 0.6498 - val_accuracy: 0.8456 - val_loss: 0.4482
Epoch 11/100
2116/2138 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7401 - loss: 0.6727

2138/2138 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.7403 - loss: 0.6726 - val_accuracy: 0.8380 - val_loss: 0.4394
Epoch 12/100
2138/2138 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.7576 - loss: 0.6546 - val_accuracy: 0.7813 - val_loss: 0.5350
Epoch 13/100
2138/2138 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.7489 - loss: 0.6656 - val_accuracy: 0.8112 - val_loss: 0.4828
Epoch 14/100
2138/2138 ━━━━━━━━━━━━━━━━━━━━ 14s 6ms/step - accuracy: 0.7651 - loss: 0.6498 - val_accuracy: 0.8055 - val_loss: 0.4897
Epoch 15/100
2138/2138 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - accuracy: 0.7765 - loss: 0.6385 - val_accuracy: 0.8209 - val_loss: 0.4614
Epoch 16/100
2138/2138 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.7724 - loss: 0.6397 - val_accuracy: 0.8148 - val_loss: 0.4746
Epoch 17/100
2138/2138 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.7808 - loss: 0.6333 - val_accuracy: 0.8065 - val_loss: 0.4887
Epoch 18/100
2138/2138 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.7792 - loss: 0.

2138/2138 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.7936 - loss: 0.6198 - val_accuracy: 0.8475 - val_loss: 0.4282
Epoch 54/100
2138/2138 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.7819 - loss: 0.6263 - val_accuracy: 0.8093 - val_loss: 0.4812
Epoch 55/100
2138/2138 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.7904 - loss: 0.6221 - val_accuracy: 0.8153 - val_loss: 0.4695
Epoch 56/100
2138/2138 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.7924 - loss: 0.6211 - val_accuracy: 0.8128 - val_loss: 0.4785
Epoch 57/100
2138/2138 ━━━━━━━━━━━━━━━━━━━━ 13s 6ms/step - accuracy: 0.7908 - loss: 0.6258 - val_accuracy: 0.8202 - val_loss: 0.4657
Epoch 58/100
2138/2138 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - accuracy: 0.7786 - loss: 0.6408 - val_accuracy: 0.8258 - val_loss: 0.4584
Epoch 59/100
2138/2138 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - accuracy: 0.7834 - loss: 0.6279 - val_accuracy: 0.8125 - val_loss: 0.4752
Epoch 60/100
2138/2138 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - accuracy: 0.7872 - loss:

1. train_inputs
(query embeddings, product embeddings)

Example:

[
  [q0...q31],   # query
  [p0...p159]   # product
]
2. train_labels
0 or 1

Example:

1 → relevant
0 → not relevant
3. epochs = 100

👉 poora dataset 100 baar dekhega

repeat learning 100 times
4. batch_size = 64

👉 ek baar me 64 samples

fast + memory efficient training
5. validation_data

👉 model ko test karta hai unseen data pe

train = learning
val = exam
6. callbacks
callbacks=[model_checkpoint]

👉 “best model save karte jao”

7. class_weight (IMPORTANT)
class_weight=dict(enumerate([1, 2]))

Matlab:

Class	Weight
0 (negative)	1
1 (positive)	2
why?

Dataset imbalance:

0 → bahut zyada (irrelevant)
1 → kam (relevant)

So model ko force karte hain:

👉 positive examples ko zyada importance do

⚡ Real production flow
🟢 OFFLINE (once)
All products → Dense → Normalize → store vectors
🔵 ONLINE (real-time)
Query → Dense → Normalize
        ↓
FAISS search (fast)
        ↓
Top-K products

| Part               | When computed |
| ------------------ | ------------- |
| Product embeddings | OFFLINE       |
| Query embeddings   | ONLINE        |
| Similarity search  | FAST (FAISS)  |


In [76]:
for layer in model.layers:
    print(layer.name)

input_product
input_layer
embedding_layer_product
dense
normalize_product
normalize_query
dot


In [78]:
product_model = Model(
    inputs=[input_query, input_product],
    outputs=model.get_layer('normalize_product').output
)
#Main full model se sirf product tower ka output nikalna chahta hoon

Tum model se sirf product embeddings nikaal rahe ho, taaki baad me fast search ho sake.

Product → 16-d vector (precompute)
Query   → 16-d vector (runtime)
Similarity → dot product (fast search)

Model ke inputs 2 hain:

input_query
input_product

Lekin output sirf:

normalize_product (16-d product embedding)

Step 3: Dummy queries kyu de rahe ho?
np.array([[0]*32] * N)

Matlab:

N products ke liye fake query = zero vector
🤔 WHY?

Kyuki model ka structure hai:

(query, product) → output

BUT tumhe sirf:

product embedding chahiye

So query ko ignore karne ke liye dummy input diya.

🧠 Real intuition

Socho model bol raha hai:

"mujhe query bhi chahiye, product bhi"

Lekin tum bol rahe ho:

"query mujhe nahi chahiye, sirf product encoding chahiye"

So tum fake query de dete ho.

In [79]:
input_data_product = [
    np.array([[0]*32] * df_prod_embed.shape[0]),  # dummy queries
    np.array(df_prod_embed[product_tower_cols])
]

product_embeddings = product_model.predict(input_data_product)

4615/4615 ━━━━━━━━━━━━━━━━━━━━ 3s 679us/step


In [80]:
product_embeddings.shape

(147673, 16)

PRODUCTS (160-d)
      ↓
product_model
      ↓
PRODUCT EMBEDDINGS (16-d)
      ↓
STORE (FAISS)

--------------------------------

QUERY (32-d)
      ↓
query_model
      ↓
QUERY EMBEDDING (16-d)
      ↓
SEARCH in FAISS

embeddings prepare karo

Step 2: DataFrame me convert karo

In [84]:
df_product_embeddings_model = pd.DataFrame(
    product_embeddings,
    columns=['p'+str(x) for x in range(16)]
)

In [85]:
df_prod_embed

,p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,...,p153,p154,p155,p156,p157,p158,p159,product_id,product_title,pid
0,0.057728,0.011255,-0.023404,0.092429,-0.020029,0.018735,0.003974,0.008556,-0.005657,0.016547,...,0.006825,0.076608,-0.019186,-0.010564,0.000239,-0.043817,-0.097729,B01A5DU2TA,"Crocs Kids' Handle It Rain Boots , Candy Pink,...",0
1,0.038126,0.004931,-0.022067,0.066682,0.009379,0.017764,0.019343,0.039371,0.001445,0.010448,...,0.021710,0.093479,-0.026546,-0.052332,0.004328,-0.019683,-0.112607,B07346RPKN,"Hatley Kids' Little Classic Rain Boots, Pink &...",1
2,0.036354,0.016651,0.012105,0.096476,0.024649,-0.013194,0.018663,0.024827,-0.021811,-0.019258,...,0.026384,0.075618,0.015224,-0.049442,0.055570,-0.058420,-0.071863,B078TCVZ4V,Outee Rubber Kids Rain Boots,2
3,0.046325,0.016477,-0.022943,0.018492,-0.025954,-0.005175,0.018342,0.041428,-0.006838,-0.008621,...,0.021611,0.051968,-0.002022,-0.055210,0.022411,-0.055057,-0.084781,B07BR3PPDR,Hope & Henry Girls' Red Milano Stitch Cardigan,3
4,0.042861,-0.012059,-0.051115,0.127641,0.026230,0.011335,0.004991,0.008443,0.037345,-0.019857,...,0.021611,0.051968,-0.002022,-0.055210,0.022411,-0.055057,-0.084781,B07H5C4P19,Spring&Gege Youth Solid Full Zipper Hoodies So...,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147668,0.052873,-0.033392,-0.078980,0.077770,-0.016582,-0.000300,0.038925,0.009066,0.018365,-0.021745,...,0.002387,0.040072,-0.002007,-0.018031,0.011338,-0.050491,-0.088851,B08DQ2HNNB,Jurassic Park T-Rex Skull 1-Ply Reusable Face ...,147668
147669,0.008746,-0.083402,0.014344,-0.000829,0.019679,-0.001146,-0.014921,0.018712,0.029838,-0.006820,...,0.002387,0.040072,-0.002007,-0.018031,0.011338,-0.050491,-0.088851,B07DQCL946,The Jurassic Games,147669
147670,0.026175,-0.062091,-0.011162,0.036014,-0.043672,0.024141,0.013321,-0.015863,-0.006382,0.024280,...,0.026201,0.039388,0.063526,0.018322,0.059560,-0.036569,-0.126742,B07FPT34TS,Jurassic+World%3a+Fallen+Kingdom+Velociraptor+...,147670
147671,0.050990,-0.006569,-0.058267,0.085894,0.017769,0.008434,0.016887,0.010712,-0.017868,-0.012275,...,0.017682,0.054370,-0.040309,-0.028475,0.002260,-0.008909,-0.113362,B07BX7PCZX,Jurassic Park 8-Bit Logo Unisex Youth Pull-Ove...,147671


In [86]:
df_product_embeddings_model

,p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11,p12,p13,p14,p15
0,0.081718,0.252420,0.130305,-0.198380,-0.297830,-0.108349,0.213237,0.069627,0.352895,0.150246,-0.149548,-0.240038,-0.110423,-0.218795,-0.212806,-0.624556
1,-0.071770,0.180067,0.170059,-0.524229,-0.246497,-0.273562,0.154583,-0.029278,0.405932,0.191186,0.009301,0.015923,-0.047215,-0.307367,-0.229712,-0.383659
2,0.101907,0.405210,0.229943,-0.286620,-0.160123,-0.336185,0.100283,0.008053,0.126029,0.210624,-0.005904,-0.192342,-0.174554,0.047321,-0.562604,-0.308393
3,0.292526,0.043214,-0.004264,0.065412,-0.387645,-0.255135,0.149323,-0.068031,0.321671,0.202850,0.127478,-0.090371,-0.234759,-0.543900,-0.093298,-0.370533
4,0.158542,0.158387,0.097936,-0.272233,-0.185360,-0.269787,0.280425,0.082185,0.308304,-0.053447,0.052081,-0.308183,0.127285,-0.553427,-0.199867,-0.339872
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147668,0.051470,0.303316,-0.112389,-0.017005,-0.117105,0.150889,0.114638,0.317280,-0.151276,-0.196858,-0.527732,0.215311,-0.151690,-0.269122,-0.132781,-0.492520
147669,-0.120400,0.025038,0.159033,-0.211124,0.196532,0.318142,-0.144015,0.383222,0.146622,0.236157,-0.426980,0.321928,-0.434037,-0.126003,0.018158,-0.199400
147670,0.034626,-0.168943,-0.095123,-0.174529,0.201599,0.321710,0.114178,0.345805,0.191543,-0.097791,-0.423992,-0.051941,0.065744,-0.234528,0.211276,-0.566843
147671,0.161085,0.307920,0.161903,-0.079505,-0.032638,0.065696,0.008158,0.188619,-0.030425,-0.213767,-0.564764,0.111918,-0.086242,-0.392262,-0.146539,-0.494764


In [88]:
df_product_embeddings_model['pid'] = df_prod_embed['pid'].values
df_product_embeddings_model['product_title'] = df_prod_embed['product_title'].values

In [89]:
df_product_embeddings_model

,p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11,p12,p13,p14,p15,pid,product_title
0,0.081718,0.252420,0.130305,-0.198380,-0.297830,-0.108349,0.213237,0.069627,0.352895,0.150246,-0.149548,-0.240038,-0.110423,-0.218795,-0.212806,-0.624556,0,"Crocs Kids' Handle It Rain Boots , Candy Pink,..."
1,-0.071770,0.180067,0.170059,-0.524229,-0.246497,-0.273562,0.154583,-0.029278,0.405932,0.191186,0.009301,0.015923,-0.047215,-0.307367,-0.229712,-0.383659,1,"Hatley Kids' Little Classic Rain Boots, Pink &..."
2,0.101907,0.405210,0.229943,-0.286620,-0.160123,-0.336185,0.100283,0.008053,0.126029,0.210624,-0.005904,-0.192342,-0.174554,0.047321,-0.562604,-0.308393,2,Outee Rubber Kids Rain Boots
3,0.292526,0.043214,-0.004264,0.065412,-0.387645,-0.255135,0.149323,-0.068031,0.321671,0.202850,0.127478,-0.090371,-0.234759,-0.543900,-0.093298,-0.370533,3,Hope & Henry Girls' Red Milano Stitch Cardigan
4,0.158542,0.158387,0.097936,-0.272233,-0.185360,-0.269787,0.280425,0.082185,0.308304,-0.053447,0.052081,-0.308183,0.127285,-0.553427,-0.199867,-0.339872,4,Spring&Gege Youth Solid Full Zipper Hoodies So...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147668,0.051470,0.303316,-0.112389,-0.017005,-0.117105,0.150889,0.114638,0.317280,-0.151276,-0.196858,-0.527732,0.215311,-0.151690,-0.269122,-0.132781,-0.492520,147668,Jurassic Park T-Rex Skull 1-Ply Reusable Face ...
147669,-0.120400,0.025038,0.159033,-0.211124,0.196532,0.318142,-0.144015,0.383222,0.146622,0.236157,-0.426980,0.321928,-0.434037,-0.126003,0.018158,-0.199400,147669,The Jurassic Games
147670,0.034626,-0.168943,-0.095123,-0.174529,0.201599,0.321710,0.114178,0.345805,0.191543,-0.097791,-0.423992,-0.051941,0.065744,-0.234528,0.211276,-0.566843,147670,Jurassic+World%3a+Fallen+Kingdom+Velociraptor+...
147671,0.161085,0.307920,0.161903,-0.079505,-0.032638,0.065696,0.008158,0.188619,-0.030425,-0.213767,-0.564764,0.111918,-0.086242,-0.392262,-0.146539,-0.494764,147671,Jurassic Park 8-Bit Logo Unisex Youth Pull-Ove...


In [90]:
product_vectors = df_product_embeddings_model[
    ['p'+str(x) for x in range(embedding_dim)]
].values.astype('float32')

In [91]:
product_vectors

array([[ 0.0817181 ,  0.25241992,  0.1303049 , ..., -0.21879515,
        -0.2128063 , -0.62455565],
       [-0.07177022,  0.1800675 ,  0.17005876, ..., -0.30736703,
        -0.22971201, -0.38365892],
       [ 0.10190699,  0.40520978,  0.22994289, ...,  0.04732075,
        -0.56260437, -0.30839348],
       ...,
       [ 0.03462557, -0.16894329, -0.09512264, ..., -0.23452833,
         0.21127602, -0.5668428 ],
       [ 0.1610848 ,  0.30792   ,  0.16190289, ..., -0.3922619 ,
        -0.14653929, -0.49476367],
       [-0.36962083,  0.07614298,  0.1766582 , ..., -0.07840764,
        -0.13819937, -0.67128485]], shape=(147673, 16), dtype=float32)

In [92]:
faiss.normalize_L2(product_vectors)

In [93]:
#index create (AnnoyIndex equivalent)
index = faiss.IndexFlatIP(embedding_dim)

In [94]:
#add vectors (like add_item)
index.add(product_vectors)

In [95]:
# pid → title mapping (same as mp_product_dict)
mp_product_dict = {}

for ix, row in df_product_embeddings_model.iterrows():
    mp_product_dict[int(row['pid'])] = row['product_title']

In [96]:
#save index (Annoy .save equivalent)
faiss.write_index(index, "product_model.faiss")

In [97]:
query_vec = np.array([query_embedding]).astype('float32')
faiss.normalize_L2(query_vec)

D, I = index.search(query_vec, k=10)

NameError: name 'query_embedding' is not defined

In [98]:
final_query_embedding

<KerasTensor shape=(None, 16), dtype=float32, sparse=False, ragged=False, name=keras_tensor_1>

In [100]:
query_model = Model(
    inputs=input_query,
    outputs=normalized_query
)

In [101]:
query_input = np.zeros((1, 32), dtype='float32')  # example

In [102]:
query_vec = query_model.predict(query_input)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 351ms/step


In [103]:
faiss.normalize_L2(query_vec)
D, I = index.search(query_vec, k=10)

In [104]:
D

array([[0.9393134 , 0.92725354, 0.910125  , 0.9037086 , 0.89645225,
        0.88669896, 0.8820734 , 0.8802621 , 0.87944865, 0.8776109 ]],
      dtype=float32)

In [105]:
I

array([[ 98418, 107565, 131613, 115889,  61389,  95274,  34565, 107336,
         64882, 123204]])